# Klasifikasi Tingkat Kemiskinan Kabupaten/Kota di Indonesia
### Perbandingan Algoritma Machine Learning untuk Mendukung SDGs 1: Tanpa Kemiskinan

**Latar belakang singkat:**
Kemiskinan adalah salah satu masalah pembangunan utama di Indonesia. SDGs (Sustainable
Development Goals) poin 1 menargetkan pengentasan kemiskinan di segala bentuk. Salah satu
cara mendukung target ini adalah dengan membangun model yang bisa mengklasifikasikan suatu
wilayah (kabupaten/kota) ke dalam kategori **Miskin** atau **Tidak Miskin**, berdasarkan
indikator sosial-ekonominya. Model semacam ini bisa membantu pemerintah memprioritaskan
wilayah mana yang perlu diberi perhatian/intervensi lebih dulu.

**Tujuan notebook ini:**
1. Membersihkan dan mengeksplorasi dataset kemiskinan 514 kabupaten/kota se-Indonesia.
2. Melatih dan membandingkan 6 algoritma klasifikasi (Decision Tree, Random Forest, Naive
   Bayes, KNN, SVM, dan Logistic Regression).
3. Memilih model dengan performa terbaik, sekaligus mengevaluasi apakah performanya "wajar"
   (mengecek potensi *data leakage*).
4. Menguji model dengan data baru yang dimasukkan secara manual.

**Dataset:** "Klasifikasi Tingkat Kemiskinan di Indonesia" (Kaggle, 514 baris, level
kabupaten/kota, bersumber dari data BPS/Susenas).

## 1. Import Library

Sebelum mulai, kita panggil dulu semua library yang dipakai sepanjang notebook ini:
- **pandas & numpy** — untuk membaca dan mengolah data tabel
- **matplotlib & seaborn** — untuk membuat grafik/visualisasi
- **scikit-learn (sklearn)** — untuk membagi data, melatih model ML, dan mengevaluasi hasil
- **imbalanced-learn (SMOTE)** — untuk mengatasi data yang tidak seimbang antar kelas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE

## 2. Memuat dan Membersihkan Data

Dataset asli diunduh dalam format CSV dengan format penulisan angka gaya Eropa: kolom
dipisah tanda titik-koma (`;`) dan desimal ditulis pakai koma (misal `18,98` bukan `18.98`).
Karena itu, saat membaca file, kita perlu memberi tahu pandas format tersebut lewat parameter
`sep=';'` dan `decimal=','`.

Setelah data terbaca, ada 3 hal yang perlu dibersihkan:
1. **Baris kosong** — file aslinya menyisakan ratusan baris kosong di bagian bawah (bekas
   template Excel), harus dibuang supaya tidak mengganggu perhitungan statistik.
2. **Dua kolom yang salah terbaca sebagai teks** — kolom *Persentase Penduduk Miskin (P0)*
   dan *Rata-rata Lama Sekolah* ternyata tetap terbaca sebagai teks (bukan angka) karena
   variasi format desimalnya. Kita perbaiki manual dengan mengganti koma menjadi titik lalu
   mengonversinya ke tipe `float`.
3. Setelah dibersihkan, seharusnya tersisa tepat **514 baris** data — sesuai jumlah asli
   kabupaten/kota yang didokumentasikan di deskripsi dataset Kaggle.

In [ ]:
df = pd.read_csv("Klasifikasi_Tingkat_Kemiskinan_di_Indonesia.csv", sep=';', decimal=',')
df.head()

In [ ]:
df = df.dropna(how='all').reset_index(drop=True)

col1 = 'Persentase Penduduk Miskin (P0) Menurut Kabupaten/Kota (Persen)'
col2 = 'Rata-rata Lama Sekolah Penduduk 15+ (Tahun)'

for c in [col1, col2]:
    df[c] = df[c].astype(str).str.replace(',', '.').astype(float)

print("Shape setelah dibersihkan:", df.shape)
df.info()

## 3. Eksplorasi Data (EDA)

Sebelum masuk ke pemodelan, penting untuk memahami karakteristik data terlebih dahulu:
- **Distribusi label**: berapa proporsi kabupaten/kota yang berlabel "Miskin" (1) vs
  "Tidak Miskin" (0)? Ini penting karena kalau proporsinya jomplang (*imbalanced*), model
  bisa "curang" dengan menebak kelas mayoritas terus dan tetap terlihat akurat padahal
  gagal mengenali kelas minoritas.
- **Statistik deskriptif** tiap variabel (rata-rata, minimum, maksimum, dll) untuk melihat
  rentang nilai wajar dari tiap indikator, dan mengecek ada tidaknya nilai aneh (outlier).

In [ ]:
print("Jumlah data per kelas:")
print(df['Klasifikasi Kemiskinan'].value_counts())
print("\nProporsi (%):")
print((df['Klasifikasi Kemiskinan'].value_counts(normalize=True) * 100).round(2))

sns.countplot(data=df, x='Klasifikasi Kemiskinan')
plt.title("Distribusi Kelas: 0 = Tidak Miskin, 1 = Miskin")
plt.show()

df.describe().round(2)

## 4. Persiapan Data Sebelum Training

Ada tiga langkah persiapan yang perlu dilakukan sebelum data siap dipakai melatih model:

**a) Memisahkan Fitur (X) dan Label (y).**
Kolom `Provinsi` dan `Kab/Kota` dibuang dari fitur karena isinya nama daerah (teks), bukan
angka, dan tidak relevan untuk dijadikan variabel prediktor. Kolom `Klasifikasi Kemiskinan`
dipisah menjadi `y` karena itu adalah target/label yang ingin diprediksi, bukan input.

**b) Menyeimbangkan Data dengan SMOTE.**
Dari EDA di atas, kita tahu datanya tidak seimbang (sekitar 88% "Tidak Miskin" vs 12%
"Miskin"). Kalau dibiarkan, model cenderung bias ke kelas mayoritas. **SMOTE (Synthetic
Minority Oversampling Technique)** mengatasi ini dengan cara membuat data sintetis baru
untuk kelas minoritas (dengan cara menginterpolasi antar data yang mirip), sehingga jumlah
kedua kelas menjadi seimbang tanpa harus membuang data kelas mayoritas.

**c) Membagi Data Latih dan Data Uji (80:20).**
Data dibagi menjadi 80% untuk melatih model (*training set*) dan 20% untuk mengujinya
(*testing set*) — ini penting agar performa model diukur dari data yang **belum pernah
dilihat model sebelumnya**, sehingga hasil evaluasinya jujur (tidak sekadar menghafal data
yang sama).

In [ ]:
X = df.drop(columns=['Provinsi', 'Kab/Kota', 'Klasifikasi Kemiskinan'])
y = df['Klasifikasi Kemiskinan']

X.head()

In [ ]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print("Sebelum SMOTE:", y.value_counts().to_dict())
print("Sesudah SMOTE :", y_resampled.value_counts().to_dict())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

print("Data latih:", X_train.shape)
print("Data uji  :", X_test.shape)

## 5. Melatih dan Membandingkan 6 Algoritma Klasifikasi

Daripada hanya mencoba satu algoritma, kita latih **enam algoritma klasifikasi sekaligus**
dan bandingkan performanya, supaya kita tahu algoritma mana yang paling cocok untuk kasus
ini:

| Algoritma | Karakteristik singkat |
|---|---|
| Decision Tree | Membuat aturan keputusan bertingkat seperti pohon, mudah diinterpretasi |
| Random Forest | Gabungan banyak Decision Tree, biasanya lebih stabil & akurat |
| Naive Bayes | Berdasarkan probabilitas, cepat dan simpel |
| KNN | Menilai data baru berdasarkan kemiripan dengan data tetangga terdekat |
| SVM | Mencari batas pemisah terbaik antar kelas |
| Logistic Regression | Model statistik klasik untuk klasifikasi biner |

Setiap model dilatih dengan data yang sama (`X_train`, `y_train`), lalu diuji dengan data
yang sama juga (`X_test`, `y_test`), supaya perbandingannya adil (*apple-to-apple*). Untuk
tiap model, kita simpan skor akurasinya dan cetak `classification_report` (berisi precision,
recall, f1-score) — bukan cuma akurasi, karena akurasi saja bisa menyesatkan pada data yang
tadinya tidak seimbang.

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=100),
    "Naive Bayes": GaussianNB(),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVM": SVC(random_state=42, probability=True),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
}

In [ ]:
results = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    predictions[name] = y_pred
    print(f"=== {name} ===")
    print(f"Akurasi: {acc:.4f}")
    print(classification_report(y_test, y_pred))
    print("-" * 50)

## 6. Memilih Model Terbaik dan Melihat Detail Performanya

Setelah semua model dilatih, langkah berikutnya adalah membandingkan akurasinya secara
berdampingan (lewat tabel dan grafik batang), lalu **secara otomatis memilih model dengan
akurasi tertinggi** sebagai kandidat model terbaik.

Untuk model terbaik itu, kita gali lebih dalam dengan dua cara:
- **Confusion Matrix** — tabel yang merinci berapa banyak prediksi benar/salah untuk
  masing-masing kelas (bukan cuma satu angka akurasi), supaya kita tahu apakah model lebih
  sering salah menebak kelas "Miskin" atau kelas "Tidak Miskin".
- **Feature Importance** — khusus untuk model berbasis pohon (Decision Tree/Random Forest),
  kita bisa melihat variabel mana yang paling berpengaruh terhadap keputusan model. Ini
  berguna untuk interpretasi kebijakan: variabel apa yang paling menentukan status kemiskinan
  suatu daerah?

In [ ]:
results_df = pd.DataFrame(list(results.items()), columns=['Model', 'Akurasi'])
results_df = results_df.sort_values(by='Akurasi', ascending=False).reset_index(drop=True)
print(results_df)

plt.figure(figsize=(8, 5))
sns.barplot(data=results_df, x='Akurasi', y='Model', palette='viridis')
plt.title("Perbandingan Akurasi Model")
plt.xlim(0, 1)
for i, v in enumerate(results_df['Akurasi']):
    plt.text(v + 0.01, i, f"{v:.4f}", va='center')
plt.show()

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_accuracy = results_df.iloc[0]['Akurasi']
best_model = models[best_model_name]
best_pred = predictions[best_model_name]

print(f"Model terbaik : {best_model_name}")
print(f"Akurasi       : {best_accuracy:.4f}")
print("\nClassification Report Model Terbaik:")
print(classification_report(y_test, best_pred))

In [ ]:
cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f"Confusion Matrix - {best_model_name}")
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.show()

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importance_df = pd.DataFrame({
        'Fitur': X.columns,
        'Importance': best_model.feature_importances_
    }).sort_values(by='Importance', ascending=False)

    plt.figure(figsize=(8, 5))
    sns.barplot(data=importance_df, x='Importance', y='Fitur', palette='mako')
    plt.title(f"Feature Importance - {best_model_name}")
    plt.show()

    print(importance_df)
else:
    print(f"Model {best_model_name} tidak memiliki atribut feature_importances_ bawaan.")

## 7. Mengecek dan Mengatasi Data Leakage

Akurasi model terbaik di atas kemungkinan besar terlihat **sangat tinggi (sekitar 98%)** —
ini justru perlu dicurigai, bukan langsung dianggap sebagai prestasi. Kecurigaan ini muncul
karena salah satu fitur input, yaitu **"Persentase Penduduk Miskin (P0)"**, kemungkinan besar
adalah **variabel yang dipakai untuk menentukan label** `Klasifikasi Kemiskinan` itu sendiri
sejak awal (misalnya lewat aturan sederhana: jika P0 di atas angka tertentu, maka label = 1).

Kalau dugaan ini benar, artinya model bukan belajar pola dari variabel-variabel sosial
ekonomi lain (IPM, pendidikan, sanitasi, dll), melainkan hanya "mengintip" langsung salah
satu sumber pembentuk labelnya sendiri. Ini disebut ***data leakage*** — kebocoran informasi
target ke dalam fitur, yang membuat performa model di atas kertas terlihat bagus, padahal
tidak benar-benar berguna untuk memprediksi kasus baru di dunia nyata (yang labelnya belum
diketahui).

**Cara mengeceknya:** kita ulangi seluruh proses (SMOTE → split data → training ke-6 model)
namun kali ini **kolom P0 dibuang dari daftar fitur**. Jika akurasi turun secara signifikan,
itu mengkonfirmasi dugaan data leakage — dan model versi **tanpa kolom P0** inilah yang harus
dipakai sebagai model final, karena lebih jujur mencerminkan kemampuan prediksi yang
sebenarnya.

In [ ]:
X_no_leak = df.drop(columns=['Provinsi', 'Kab/Kota', 'Klasifikasi Kemiskinan',
                               'Persentase Penduduk Miskin (P0) Menurut Kabupaten/Kota (Persen)'])
y_no_leak = df['Klasifikasi Kemiskinan']

X_resampled2, y_resampled2 = smote.fit_resample(X_no_leak, y_no_leak)

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_resampled2, y_resampled2, test_size=0.2, random_state=42, stratify=y_resampled2
)

print("Jumlah fitur sebelumnya (dengan P0):", X.shape[1])
print("Jumlah fitur sekarang (tanpa P0)  :", X_no_leak.shape[1])

In [ ]:
results_no_leak = {}
predictions_no_leak = {}

for name, model in models.items():
    model.fit(X_train2, y_train2)
    y_pred2 = model.predict(X_test2)
    acc2 = accuracy_score(y_test2, y_pred2)
    results_no_leak[name] = acc2
    predictions_no_leak[name] = y_pred2
    print(f"=== {name} (tanpa P0) ===")
    print(f"Akurasi: {acc2:.4f}")
    print(classification_report(y_test2, y_pred2))
    print("-" * 50)

In [ ]:
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Akurasi (dengan P0)': list(results.values()),
    'Akurasi (tanpa P0)': [results_no_leak[m] for m in results.keys()]
}).sort_values(by='Akurasi (tanpa P0)', ascending=False).reset_index(drop=True)

comparison_df

## 8. Menetapkan Model Final dan Menguji dengan Data Baru

Berdasarkan tabel perbandingan di atas, model final yang dipilih adalah algoritma dengan
akurasi tertinggi **setelah kolom P0 dibuang** (bukan lagi dari perbandingan awal yang masih
mengandung kebocoran data). Model final ini kemudian dilatih ulang secara eksplisit
menggunakan seluruh data latih versi bersih (`X_train2`, `y_train2`).

Untuk menguji apakah model bekerja secara masuk akal, kita sediakan fitur **input manual**:
pengguna (misalnya dosen atau siapa pun yang menguji notebook ini) bisa mengetikkan sendiri
nilai-nilai indikator suatu daerah hipotetis, lalu program akan menampilkan prediksi model
beserta tingkat keyakinannya (probabilitas) untuk masing-masing kelas.

In [ ]:
final_model = RandomForestClassifier(random_state=42, n_estimators=100)
final_model.fit(X_train2, y_train2)

fitur_list = list(X_no_leak.columns)
print("Fitur yang perlu diisi saat prediksi:")
for i, f in enumerate(fitur_list, 1):
    print(f"{i}. {f}")

In [ ]:
print("=== Masukkan Data untuk Prediksi Kemiskinan ===\n")

input_values = {}
for fitur in fitur_list:
    while True:
        try:
            nilai = float(input(f"{fitur}: "))
            input_values[fitur] = nilai
            break
        except ValueError:
            print("   -> Input harus berupa angka, coba lagi.")

data_user = pd.DataFrame([input_values])
data_user = data_user[fitur_list]

prediksi = final_model.predict(data_user)
probabilitas = final_model.predict_proba(data_user)

label_map = {0: "Tidak Miskin", 1: "Miskin"}
print("\n=== HASIL PREDIKSI ===")
print(f"Status: {label_map[prediksi[0]]}")
print(f"Probabilitas Tidak Miskin: {probabilitas[0][0]:.2%}")
print(f"Probabilitas Miskin      : {probabilitas[0][1]:.2%}")

## 9. Kesimpulan

Isi bagian ini berdasarkan hasil aktual yang kamu dapatkan di atas:

- **Model terbaik (versi bersih, tanpa data leakage):** _isi nama model & akurasinya dari
  tabel `comparison_df`, kolom "Akurasi (tanpa P0)"_
- **Seberapa besar akurasi turun setelah P0 dibuang:** _bandingkan angka "dengan P0" vs
  "tanpa P0" — semakin besar penurunannya, semakin kuat indikasi bahwa P0 memang sumber
  kebocoran data_
- **Fitur paling berpengaruh terhadap status kemiskinan:** _lihat hasil feature importance
  di bagian 6, misalnya IPM, pengeluaran per kapita, atau akses sanitasi_
- **Kaitan dengan SDGs 1 (Tanpa Kemiskinan):** jelaskan bagaimana model semacam ini bisa
  membantu pemerintah mengidentifikasi wilayah prioritas untuk program pengentasan
  kemiskinan, berdasarkan variabel yang paling berpengaruh tadi.